In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("exam-prep").master("local[*]").getOrCreate()
print(spark.version)

4.3.0


In [3]:
from pyspark.sql import functions as F, Window

In [4]:


spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")  # parses dates like "09-NOV-25"

WEATHER_CODES = ["X1","X2","X4","X9","XJ","XT","XW","QH","QI","QJ",
                 "IW","IP","JT","JH","JK","J6","OG"]

# Delays -> weather delay minutes per day
delays = (spark.read.option("header", True).option("inferSchema", True)
          .csv("../data/raw/Delays/Transparency 25-26 P04.csv"))

daily = (delays.filter(F.col("INCIDENT_REASON").isin(WEATHER_CODES))
         .withColumn("date", F.to_date("ORIGIN_DEPARTURE_DATE", "dd-MMM-yy"))
         .groupBy("date").agg(F.sum("PFPI_MINUTES").alias("weather_min")))

# Weather -> add a 3-day rolling rain feature per city (on the full history, before joining)
w = Window.partitionBy("city").orderBy("date").rowsBetween(-2, 0)
weather = (spark.read.option("header", True).option("inferSchema", True)
           .csv("../data/raw/weather_daily.csv")
           .withColumn("date", F.to_date("time"))
           .withColumn("rain_3d", F.sum("precipitation_sum").over(w)))

data = weather.join(daily, "date")   # keeps only the P04 dates
print(data.count(), "rows")          # expect ~28 days x 8 cities = 224

224 rows


In [5]:
cutoff = "2025-07-13"
train = data.filter(F.col("date") < cutoff)    # ~21 days
test  = data.filter(F.col("date") >= cutoff)   # last ~7 days

# Threshold from TRAINING days only (one value per day, hence distinct)
threshold = (train.select("date", "weather_min").distinct()
             .approxQuantile("weather_min", [0.75], 0)[0])
print("High-disruption threshold (minutes):", threshold)

add_label = lambda df: df.withColumn("label", (F.col("weather_min") >= threshold).cast("double"))
train, test = add_label(train), add_label(test)

High-disruption threshold (minutes): 2403.0


In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

features = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum",
            "rain_3d", "wind_gusts_10m_max"]

indexer   = StringIndexer(inputCol="city", outputCol="city_idx", handleInvalid="keep")
encoder   = OneHotEncoder(inputCols=["city_idx"], outputCols=["city_ohe"])
assembler = VectorAssembler(inputCols=features + ["city_ohe"], outputCol="features")
lr        = LogisticRegression(featuresCol="features", labelCol="label")

pipeline = Pipeline(stages=[indexer, encoder, assembler, lr])
model = pipeline.fit(train)        # every estimator learns from TRAIN only
preds = model.transform(test)      # the fitted PipelineModel just applies them

26/11/27 22:42:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [7]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

ev = BinaryClassificationEvaluator(labelCol="label")      # default metric = areaUnderROC
print("areaUnderROC:", round(ev.evaluate(preds), 3))
print("areaUnderPR: ", round(ev.evaluate(preds, {ev.metricName: "areaUnderPR"}), 3))

(preds.withColumn("p_disrupt", vector_to_array("probability")[1])
      .groupBy("date")
      .agg(F.first("label").alias("actual"), F.round(F.avg("p_disrupt"), 2).alias("avg_predicted_p"))
      .orderBy("date").show())

areaUnderROC: 0.445
areaUnderPR:  0.128
+----------+------+---------------+
|      date|actual|avg_predicted_p|
+----------+------+---------------+
|2025-07-13|   0.0|            0.7|
|2025-07-14|   0.0|           0.39|
|2025-07-15|   1.0|           0.43|
|2025-07-16|   0.0|           0.03|
|2025-07-17|   0.0|           0.32|
|2025-07-18|   0.0|           0.78|
|2025-07-19|   0.0|           0.72|
+----------+------+---------------+



In [8]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(featuresCol="features", labelCol="label",
                    maxIter=50, maxDepth=3, seed=42)
gbt_pipeline = Pipeline(stages=[indexer, encoder, assembler, gbt])

gbt_model = gbt_pipeline.fit(train)
gbt_preds = gbt_model.transform(test)

print("GBT areaUnderROC:", round(ev.evaluate(gbt_preds), 3))
print("GBT areaUnderPR: ", round(ev.evaluate(gbt_preds, {ev.metricName: "areaUnderPR"}), 3))

(gbt_preds.withColumn("p_disrupt", vector_to_array("probability")[1])
          .groupBy("date")
          .agg(F.first("label").alias("actual"),
               F.round(F.avg("p_disrupt"), 2).alias("avg_predicted_p"))
          .orderBy("date").show())

GBT areaUnderROC: 0.204
GBT areaUnderPR:  0.085
+----------+------+---------------+
|      date|actual|avg_predicted_p|
+----------+------+---------------+
|2025-07-13|   0.0|            0.8|
|2025-07-14|   0.0|           0.29|
|2025-07-15|   1.0|           0.16|
|2025-07-16|   0.0|           0.07|
|2025-07-17|   0.0|           0.23|
|2025-07-18|   0.0|           0.46|
|2025-07-19|   0.0|           0.68|
+----------+------+---------------+



In [9]:
imp = gbt_model.stages[-1].featureImportances
names = assembler.getInputCols()
print(features)          # the 5 weather features
print(imp)               # importance per vector slot (weather first, then city one-hot slots)

['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'rain_3d', 'wind_gusts_10m_max']
(13,[0,1,2,3,4,6,7,9,10],[0.1795651519304575,0.07532431270644278,0.16537089385819603,0.281868607636983,0.21815944599140105,0.0021144273582120576,0.04154430766487298,0.003163527530077795,0.03288932532335698])


In [10]:
# base data

from pyspark.sql import functions as F

base = (train.withColumn("split", F.lit("train"))
        .unionByName(test.withColumn("split", F.lit("test")))
        .select("split", "date", "city",
                "temperature_2m_max", "temperature_2m_min",
                "precipitation_sum", "rain_3d", "wind_gusts_10m_max",
                "weather_min", "label")
        .orderBy("date", "city"))

print("Rows:", base.count(), "| Label threshold (minutes):", threshold)
base.show(300, truncate=False)

Rows: 224 | Label threshold (minutes): 2403.0
+-----+----------+----------+------------------+------------------+-----------------+------------------+------------------+-----------+-----+
|split|date      |city      |temperature_2m_max|temperature_2m_min|precipitation_sum|rain_3d           |wind_gusts_10m_max|weather_min|label|
+-----+----------+----------+------------------+------------------+-----------------+------------------+------------------+-----------+-----+
|train|2025-06-22|Birmingham|21.6              |14.5              |5.2              |10.600000000000001|63.7              |639.0      |0.0  |
|train|2025-06-22|Bristol   |20.4              |15.6              |3.6              |4.8               |56.9              |639.0      |0.0  |
|train|2025-06-22|Edinburgh |18.1              |10.6              |2.4              |8.1               |64.4              |639.0      |0.0  |
|train|2025-06-22|Glasgow   |16.0              |10.1              |6.1              |10.0             

In [13]:
base.filter("split = 'train'").show(truncate=False)

+-----+----------+----------+------------------+------------------+-----------------+------------------+------------------+-----------+-----+
|split|date      |city      |temperature_2m_max|temperature_2m_min|precipitation_sum|rain_3d           |wind_gusts_10m_max|weather_min|label|
+-----+----------+----------+------------------+------------------+-----------------+------------------+------------------+-----------+-----+
|train|2025-06-22|Birmingham|21.6              |14.5              |5.2              |10.600000000000001|63.7              |639.0      |0.0  |
|train|2025-06-22|Bristol   |20.4              |15.6              |3.6              |4.8               |56.9              |639.0      |0.0  |
|train|2025-06-22|Edinburgh |18.1              |10.6              |2.4              |8.1               |64.4              |639.0      |0.0  |
|train|2025-06-22|Glasgow   |16.0              |10.1              |6.1              |10.0              |58.3              |639.0      |0.0  |
|train

In [15]:
delays_raw  = spark.read.option("header", True).option("inferSchema", True).csv("../data/raw/Delays/Transparency 25-26 P04.csv")
weather_raw = spark.read.option("header", True).option("inferSchema", True).csv("../data/raw/weather_daily.csv")

print("Delays:", delays_raw.count(), "rows");   delays_raw.show(25, truncate=False)
print("Weather:", weather_raw.count(), "rows"); weather_raw.show(25, truncate=False)

Delays: 465703 rows
+---------------------+---------------------+--------------+----------------------------+---------------------------+----------------------------+--------------------------+-------------------------+--------------------------+------------------+------------------+--------+----------------+-------------------------+-------------------+-------------+-------------+-----------+----------+---------------+--------------------+-----------------------+---------------------+------------+-------------------+-------------------+---------------+-------------------+------------------+------------------------------+------------+-------------------+----------+-----------+----------+------------+----------+-----------------+------------+----------------+
|FINANCIAL_YEAR_PERIOD|ORIGIN_DEPARTURE_DATE|TRUST_TRAIN_ID|PLANNED_ORIGIN_LOCATION_CODE|PLANNED_ORIGIN_WTT_DATETIME|PLANNED_ORIGIN_GBTT_DATETIME|PLANNED_DEST_LOCATION_CODE|PLANNED_DEST_WTT_DATETIME|PLANNED_DEST_GBTT_DATETIME|TRAIN